In [1]:
%load_ext autoreload
%autoreload 2

### Match stations, using station name, lat, lon. The name matching in Raw data meta_data form, in the folder and in the database

In [2]:
from crmprtd import Row
import logging
import os
import pickle
import pandas as pd
import numpy as np
from natsort import natsorted
from natsort import natsort_keygen
from pprint import pprint
from collections import Counter
from collections import defaultdict

import re
from rapidfuzz import fuzz
from crmprtd import infer
import sqlalchemy as sa

from fern_func import *
# from sql_func import *
from fern_raw_db_dompare import *

In [3]:
# --- Main workflow ---
# --- read data ---
meta_path = '/fern_data/FERNNorth2024_VF/20241125 MeteorologicalNetworks-FERN-VF-shared.xlsx'

df_station = pd.read_excel(meta_path)
df_station = df_station.sort_values(by='station_name', key=natsort_keygen())


# --- output folder ---
output_folder = '/workspaces/crmprtd/fern/v5_after_metadata_update_rerun_testnode/output/'
os.makedirs(output_folder, exist_ok=True)


NotADirectoryError: [Errno 20] Not a directory: '/fern_data/FERNNorth2024_VF/20241125 MeteorologicalNetworks-FERN-VF-shared.xlsx'

### Fern raw data condition summary

In [ ]:
data_path = '/fern_data/FERNNorth2024_VF/WxData24'
csv_files = [f for f in os.listdir(data_path) if f.endswith('.csv')]
# Sort in natural order
csv_files = natsorted(csv_files)

summary_fern_raw_records = []

for station_i, csv_file in enumerate(csv_files):
    station_row = df_station.iloc[station_i]
    station_name = station_row[ 'station_name']

    print(station_name)

    ### summarize raw data
    records = summarize_raw_station(csv_file, station_row, data_path)
    summary_fern_raw_records.extend(records)  # add to the master list

# Create summary DataFrame
raw_summary = pd.DataFrame(summary_fern_raw_records)

raw_summary_path = os.path.join(output_folder, "Fern_raw_station_variable_summary.csv")
raw_summary.to_csv(raw_summary_path, index=False)
print(f"✅ Summary saved to {raw_summary_path}")


NotADirectoryError: [Errno 20] Not a directory: '/fern_data/FERNNorth2024_VF/WxData24'

In [ ]:
pd.DataFrame(raw_summary['station_name'].unique())

,0
0,Atlin School
1,BarrenWx
2,BlackhawkWx
3,BoulderWx
4,BowronPit
5,BulkleyWx
6,CPFWx
7,Canoe Mountain Stn
8,Caribou Pass
9,ChapmanWx


### Database summary

In [16]:
import sqlalchemy as sa
from sqlalchemy.orm import Session

engine = sa.create_engine("postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)


In [17]:
station_name_id = df_station[['native_id', 'station_name']]
native_ids = station_name_id['native_id'].tolist()
station_names = station_name_id['station_name'].tolist()

query = sa.text("""
    SELECT DISTINCT m.station_name, s.native_id
    FROM meta_history AS m
    JOIN meta_station AS s ON m.station_id = s.station_id
    WHERE s.network_id = 11
      AND m.station_name = ANY(:station_names)
""")

with engine.connect() as conn:
    existing_stations = pd.read_sql(query, conn, params={"station_names": station_names})

print("Stations found in DB:")
existing_stations

Stations found in DB:


,station_name,native_id
0,Atlin School,AtlSch
1,BarrenWx,Barren
2,BlackhawkWx,Blackhawk
3,BoulderWx,Boulder Creek
4,BowronPit,BowronPit
5,BulkleyWx,PGTIS1
6,Canoe Mountain Stn,Canoe Mountain Stn
7,Caribou Pass,CaribouPass
8,ChapmanWx,Chapman
9,ChiefLakeWx,ChiefWx


- Station `Kluskus`  
    - `native_id` in Ted's sheet: `BednestiWx`  
    - `native_id` in Vanessa's sheet: `SBSmc3Wx`

And we didn't update it according to Vanessa's sheet, so we based on station_name to match stations. All stations can be matched

In [18]:
existing_native_ids = existing_stations['native_id'].tolist()

query = sa.text("""
    SELECT 
        m.station_name,
        s.native_id,
        m.lat,
        m.lon,
        m.elev,
        v.net_var_name, 
        v.unit,
        MIN(o.obs_time) AS earliest_time,
        MAX(o.obs_time) AS latest_time
    FROM obs_raw AS o
    JOIN meta_history AS m ON o.history_id = m.history_id
    JOIN meta_vars AS v ON o.vars_id = v.vars_id
    JOIN meta_station AS s ON m.station_id = s.station_id
    WHERE s.network_id = 11
      AND s.native_id = ANY(:native_ids)
    GROUP BY v.net_var_name, v.unit, m.station_name, m.lat, m.lon, s.native_id, m.elev
    ORDER BY v.net_var_name;
""")

with engine.connect() as conn:
    df_obs = pd.read_sql(query, conn, params={"native_ids": existing_native_ids})

df_obs

,station_name,native_id,lat,lon,elev,net_var_name,unit,earliest_time,latest_time
0,Atlin School,AtlSch,59.574000,-133.687000,705.0,DewPtC,celsius,2011-07-28 09:00:00,2024-08-10 19:00:00
1,BarrenWx,Barren,54.510444,-126.614341,1051.0,DewPtC,celsius,2021-07-09 12:00:00,2024-10-03 11:00:00
2,BlackhawkWx,Blackhawk,55.078885,-120.352171,962.0,DewPtC,celsius,2012-05-24 13:00:00,2025-01-07 10:00:00
3,BoulderWx,Boulder Creek,55.108173,-127.374740,385.0,DewPtC,celsius,2010-06-07 13:00:00,2024-08-07 08:00:00
4,BowronPit,BowronPit,53.902111,-122.015389,722.0,DewPtC,celsius,2018-08-29 15:00:00,2025-01-07 11:00:00
...,...,...,...,...,...,...,...,...,...
397,SeebachWx,Seebach,54.355000,122.058000,780.0,WindSpeedms,m s-1,2023-07-13 12:00:00,2024-09-26 10:11:00
398,Sunbeam,Mcbride Mountain Stn,53.338690,-120.121080,2000.0,WindSpeedms,m s-1,2009-09-09 16:00:00,2025-01-09 08:00:00
399,Tamarac,Bednesti,53.871334,-123.370441,865.0,WindSpeedms,m s-1,2021-07-29 13:00:00,2024-10-10 12:00:00
400,ThompsonWx,Thompson,54.333115,-126.099445,869.0,WindSpeedms,m s-1,2007-09-12 16:00:00,2024-09-24 10:00:00


In [19]:
db_summary_path = os.path.join(output_folder, "Fern_db_station_variable_summary.csv")
df_obs.to_csv(db_summary_path, index=False)
print(f"✅ Summary saved to {db_summary_path}")

df_obs

✅ Summary saved to /workspaces/crmprtd/fern/v5_after_metadata_update_rerun_testnode/output/Fern_db_station_variable_summary.csv


,station_name,native_id,lat,lon,elev,net_var_name,unit,earliest_time,latest_time
0,Atlin School,AtlSch,59.574000,-133.687000,705.0,DewPtC,celsius,2011-07-28 09:00:00,2024-08-10 19:00:00
1,BarrenWx,Barren,54.510444,-126.614341,1051.0,DewPtC,celsius,2021-07-09 12:00:00,2024-10-03 11:00:00
2,BlackhawkWx,Blackhawk,55.078885,-120.352171,962.0,DewPtC,celsius,2012-05-24 13:00:00,2025-01-07 10:00:00
3,BoulderWx,Boulder Creek,55.108173,-127.374740,385.0,DewPtC,celsius,2010-06-07 13:00:00,2024-08-07 08:00:00
4,BowronPit,BowronPit,53.902111,-122.015389,722.0,DewPtC,celsius,2018-08-29 15:00:00,2025-01-07 11:00:00
...,...,...,...,...,...,...,...,...,...
397,SeebachWx,Seebach,54.355000,122.058000,780.0,WindSpeedms,m s-1,2023-07-13 12:00:00,2024-09-26 10:11:00
398,Sunbeam,Mcbride Mountain Stn,53.338690,-120.121080,2000.0,WindSpeedms,m s-1,2009-09-09 16:00:00,2025-01-09 08:00:00
399,Tamarac,Bednesti,53.871334,-123.370441,865.0,WindSpeedms,m s-1,2021-07-29 13:00:00,2024-10-10 12:00:00
400,ThompsonWx,Thompson,54.333115,-126.099445,869.0,WindSpeedms,m s-1,2007-09-12 16:00:00,2024-09-24 10:00:00
